# RLHF for Reddit TL;DR — Kaggle runner

**Before running anything:** right sidebar → *Session options* → **Accelerator: GPU T4 x1**, **Internet: On**.
Optional: *Add-ons → Secrets* → add `HF_TOKEN` (Hugging Face **write** token) and tick it.

Every cell is safe to re-run. Each stage restores what it needs from the Hugging Face Hub, so a stage
can run in a fresh session. Checkpoints and `results/` are backed up to the Hub after every stage.
For the long stages use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h.

In [ ]:
# ── 0. Config ────────────────────────────────────────────────────────────────
HF_USER   = "latent-hassan"                         # Hugging Face username (GitHub is MeMorphLing)
REPO_URL  = "https://github.com/MeMorphLing/rlhf-tldr.git"
CKPT_REPO = HF_USER + "/smollm2-135m-tldr-{stage}"      # one model repo per stage
RESULTS_REPO = HF_USER + "/rlhf-tldr-results"           # private dataset repo: backup of results/
MICRO_BATCH = ""                                        # set "8" if a stage runs out of GPU memory

import os, sys, subprocess, socket, urllib.request, shutil
BASE = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
WORK = os.path.join(BASE, "rlhf-tldr")

def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, and STOP the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (WORK if os.path.isdir(WORK) else BASE),
                         env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
print("work dir:", WORK)

In [ ]:
# ── 1. Environment check: GPU + internet (fails fast with a clear message) ─────
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("GPU:", gpu or "NONE")

net_ok = {}
for host in ["huggingface.co", "github.com", "www.kaggle.com"]:
    try:
        ip = socket.gethostbyname(host)
        code_ = urllib.request.urlopen(f"https://{host}", timeout=15).status
        net_ok[host] = True
        print(f"OK   {host:16} {ip:16} HTTP {code_}")
    except Exception as e:
        net_ok[host] = False
        print(f"FAIL {host:16} {type(e).__name__}: {e}")

if not gpu:
    raise SystemExit("No GPU. Session options → Accelerator → GPU T4 x1, then stop and start a NEW session.")
if not (net_ok["huggingface.co"] and net_ok["github.com"]):
    raise SystemExit("No internet. Session options → Internet → On, then stop and start a NEW session "
                     "(a kernel restart is not enough).")
print("\nEnvironment OK")

In [ ]:
# ── 2. Hugging Face login (Kaggle secret → fallback: paste the token) ─────────
from huggingface_hub import login, whoami
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        print("token: from Kaggle secret")
    except Exception as e:
        print(f"Kaggle secret unavailable ({type(e).__name__}) → paste the token instead")
        from getpass import getpass
        token = getpass("Hugging Face write token: ").strip()
os.environ["HF_TOKEN"] = token
login(token=token, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

In [ ]:
# ── 3. Code: clone (or update) the repo + missing dependency ──────────────────
if os.path.isdir(os.path.join(WORK, ".git")):
    sh("git pull --ff-only", cwd=WORK)
else:
    sh(f"git clone {REPO_URL} {WORK}", cwd=BASE)
os.chdir(WORK)
sh("pip install -q rouge-score")      # torch/transformers/datasets are preinstalled — don't reinstall them
import transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)

In [ ]:
# ── 4. Hub helpers: back up / restore checkpoints and results ─────────────────
from huggingface_hub import HfApi, snapshot_download
api = HfApi()
print("Hub targets:", CKPT_REPO, "|", RESULTS_REPO)

def push_ckpt(stage):
    repo = CKPT_REPO.format(stage=stage)
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=f"checkpoints/{stage}", commit_message=f"{stage} checkpoint")
    print(f"pushed checkpoints/{stage} → https://huggingface.co/{repo}")

def _ckpt_present(stage):
    return os.path.isfile(f"checkpoints/{stage}/config.json") or os.path.isfile(f"checkpoints/{stage}/reward_model.pt")

def pull_ckpt(stage):
    if _ckpt_present(stage):
        print(f"checkpoints/{stage}: already here"); return
    repo = CKPT_REPO.format(stage=stage)
    if not api.repo_exists(repo):
        raise RuntimeError(f"checkpoints/{stage} missing locally and {repo} does not exist on the Hub — run that stage first")
    snapshot_download(repo, local_dir=f"checkpoints/{stage}")
    assert _ckpt_present(stage), f"download of {repo} incomplete"
    print(f"checkpoints/{stage}: restored from {repo}")

def push_results(msg):
    api.create_repo(RESULTS_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=RESULTS_REPO, repo_type="dataset", folder_path="results", path_in_repo="results",
                      commit_message=msg)
    print(f"results/ backed up → {RESULTS_REPO}")

def pull_results():
    if not api.repo_exists(RESULTS_REPO, repo_type="dataset"):
        print(f"no results backup yet ({RESULTS_REPO}) — starting fresh"); return
    snapshot_download(RESULTS_REPO, repo_type="dataset", local_dir=".", allow_patterns="results/*")
    print(f"results/ restored from {RESULTS_REPO}")

def run_stage(target):
    sh(f"make {target}", env={"MICRO_BATCH": MICRO_BATCH} if MICRO_BATCH else None)

pull_results()

## Task 1 + dry run (run interactively first — must pass before any long run)

In [ ]:
# ── 5. Task 1 ───────────────────────────────────────────────────────────────
sh("make starter")
sh("make task1")
push_results("task 1: data stats + mask check")

In [ ]:
# ── 6. Smoke test: every stage on tiny slices with the REAL model and data (~5 min) ──
sh("make smoke")
sh("make clean")          # wipes the smoke artefacts (keeps the data cache)
pull_results()            # restore task-1 outputs that `make clean` removed
print("\nSMOKE TEST PASSED — safe to start the long runs")

## Stages (each cell can run in a fresh session)

In [ ]:
# ── 7. Stage 1: SFT ─────────────────────────────────────────────────────────
run_stage("sft")
push_ckpt("sft"); push_results("stage 1: SFT")

In [ ]:
# ── 8. Stage 2: reward model (needs SFT) ────────────────────────────────────
pull_ckpt("sft")
run_stage("reward")
push_ckpt("reward"); push_results("stage 2: reward model")

In [ ]:
# ── 9. Stage 4: DPO (needs SFT) ─────────────────────────────────────────────
pull_ckpt("sft")
run_stage("dpo")
push_ckpt("dpo"); push_results("stage 4: DPO")

In [ ]:
# ── 10. Stage 3: PPO (needs SFT + reward) — resumes automatically if a resume state exists ──
pull_ckpt("sft"); pull_ckpt("reward")
os.environ["RESUME"] = "1" if os.path.exists("checkpoints/ppo_resume/state.pt") else "0"
run_stage("ppo")
push_ckpt("ppo"); push_results("stage 3: PPO")

## Evaluation + artefacts

In [ ]:
# ── 11. Task 6 + sheets + samples + Task-7 numbers + figures ─────────────────
for s in ("sft", "reward", "ppo", "dpo"):
    pull_ckpt(s)
sh("make eval")
sh("make human-sheet")
sh("make samples")
sh("make analysis")
sh("make figures")
push_results("task 6: evaluation")
shutil.make_archive(os.path.join(BASE, "results"), "zip", WORK, "results")
print("download results.zip from the Output tab →", os.path.join(BASE, "results.zip"))

In [ ]:
# ── 12. Quick look ───────────────────────────────────────────────────────────
from IPython.display import Markdown, Image, display
display(Markdown(open("results/tables.md").read()))
for f in sorted(os.listdir("results/figures")):
    if f.endswith(".png"):
        display(Image(filename=f"results/figures/{f}", width=900))
print(open("results/correctness_checks.txt").read())